<h1 style="text-align:center">Reverse Engineering</h1>
<h2 style="text-align:center">Dynamic Analysis</h1>

<hr style="border:1px solid white">

### 1 - What is Dynamic Analysis?
Whereas static analysis looks to understand a program by extracting information directly from its bytes (using a variety of tooling), dynamic analysis observes a program mid-execution, extracting information from changes in runtime state. The greatest tool a reverse engineer has&mdash;second to their brain&mdash;is their CPU thanks to its ability to execute instructions. While static analysis has its place, code was intended to *run*, and we can learn a lot about how it works by watching it do so.

This notebook serves as an introduction to dynamic analysis, and teaches tools such as GDB and symbol preloading as they pertain to reverse engineering. A basic understanding of static analysis and the use of decompilers is strongly recommended. It will demonstrate how to solve a provided challenge in a few different ways, and students are encouraged to try to solve it themselves before reading through the full notebook. 

<hr style="border:1px solid white">

### 2 - A First Look
We are given a binary, `chal1`. Before proceeding any further, we should figure out what sort of file it is to know how to analyze the file. To figure out what sort of file a file is, we use the `file` command:

In [ ]:
!file ./chal1

Technically, `file` performs a static analysis function: it reads the bytes at the start of the binary to identify it as an Executable Linux File (ELF), then parses the ELF header to get more information&mdash;but at no point does it run the binary. The important thing here is that the file is an *executable*. This means we can *execute* (run) it, assuming we are on an x86-64 Linux system. And if we can run a binary, then we can perform dynamic analysis.

Note that this will not always be the case. Oftentimes, we may want to analyze binaries built for other architectures, or built to run in a particular execution environment, such as embedded, bare-metal firmware. The art of getting such binaries to run on our local device such that we may dynamically analyze them is called **rehosting**, and will not be covered in this lesson.

Luckily, we *can* run this binary, and as is good practice with any executable file downloaded from the internet<sup>[citation needed]</sup>, we should.

In [ ]:
!./chal1

Instantly, we are told our input was incorrect. As this is a CTF challenge, there are a few key assumptions we can make:
- we are looking for a flag,
- the program takes an input,
- the input is probably a flag,
- and the program will tell us if our flag is correct.

Such a program is commonly referred to as a "crack-me", on account that we are trying to crack the program to find the input it considers correct. Since it doesn't take any input over `stdin` (the terminal input at run-time), it likely expects a command-line argument passed in directly from the shell.

In [ ]:
!./chal1 flag{test}

Expectedly, our test input is also incorrect, but it should hopefully demonstrate the way in which we are expected to interact with the challenge.

At this point, you may want to reach for a decompiler (especially if you have completed the static analysis notebook), but we will show why that might not be the best approach.

<hr style="border:1px solid white">

### 3 - Is Ghidra Broken?
Opening it in Ghidra and using the default analyzers, we find a few functions, including the usual `entry`:

![The Ghidra Symbol Tree showing _DT_INIT, _FINI_0, _INIT_0, _INIT_1, entry, FUN_00101020, FUN_001010b0, FUN_001010e0, and FUN_00101179](img/function-list.png)

As normal, `entry` seems to call `__libc_start_main`:
```c
void processEntry entry(undefined8 param_1,undefined8 param_2)

{
  undefined1 auStack_8 [8];
  
  __libc_start_main(FUN_00101179,param_2,&stack0x00000008,0,0,param_1,auStack_8);
  do {
                    /* WARNING: Do nothing block with infinite loop */
  } while( true );
}
```

We can guess that `FUN_00101179` is likely main, and take a look at it...
```c
void FUN_00101179(undefined8 param_1,long param_2)

{
  puts("There is no flag here.");
  sleep(1);
  puts("There never was a flag here.");
  sleep(1);
  puts("Go look elsewhere.");
  sleep(1);
  puts("Or give up.");
  sleep(1);
  puts("You could also give up.");
  sleep(1);
  strcmp(*(char **)(param_2 + 8),"flag{not here}");
  return;
}
```
...but this is where things get weird.

Clearly this is not the function that gets executed when we run the program. At no point do we see any of these strings printed out, nor does the program take 5 seconds to run. Ghidra isn't wrong&mdash;these are in fact the instructions stored *statically* in the binary&mdash;but we may want to take a closer look at the *runtime* behavior instead. 

###### Note that Ida and Binary Ninja will give similar results to Ghidra's.

<hr style="border:1px solid white">

### 4 - GDB
To analyze a program as it runs (that is, *dynamically*), we use a tool called a **debugger**. The main features of a debugger typically include:
- stepping through a program one instruction at a time
- stepping through a program one line at a time (given we have the original source)
- breaking (pausing the program) when the program reaches a certain instruction/function/line
- breaking when a register or memory address changes
- inspecting the program state (registers and memory) at a certain point
- changing the program state at a certain point
- and many more features.

One such debugger is GDB (the GNU Debugger). While others exist for specific use cases, GDB is one of the most universally aplicable debuggers for its compatibility with a variety of platforms and architectures, and its powerful command-line interface. Still, its syntax is inconsistent, and its functionality poorly documented. We will present a few commands in the course of this notebook, but we recommend you find a good [reference sheet](https://raw.githubusercontent.com/hellogcc/100-gdb-tips/master/refcard.pdf) and practice using GDB until it becomes second-nature.

Open the challenge in GDB by running `gdb ./chal1` in your terminal. This will load the binary and open GDB's command environment, but not yet run the binary. We want to figure out what code actually runs as main, and we have a few different options here:
1. Run the program and break on the first instruction with `starti`. Due to the amount of start-up code from glibc, our main function is actually quite far away from the first instruction that gets executed, so this technique will not work. 
2. Add a breakpoint to the address we identified as being `main` with `b *0x101179`. Because of address-space layout randomization (ASLR), the static address of 0x101179 (`main`) will get mapped to a different address at runtime. GDB helpfully makes that address deterministic, so we could figure it out if necessary, but an easier way exists.
3. Add a first breakpoint on `__libc_start_main`, then a second breakpoint on its first argument once we reach it. For dynamically linked glibc binaries such as this one, this option will almost always work.

To add a breakpoint on `__libc_start_main`, run `b __libc_start_main`. Although this binary is stripped, we still have symbols for any functions loaded from shared libraries&mdash;thus why we can break on `__libc_start_main`, but not on `main` directly.

Next, run the binary with `r flag{test}`. Whatever we put after `r` (in this case, `flag{test}`) will be treated as the program's argument list (`argv`).

The program should immediately pause execution when it hits `__libc_start_main`:
```c
Breakpoint 1, 0x00007ffff7c277fc in __libc_start_main () from /usr/lib/libc.so.6
```
We know that the first argument of `__libc_start_main` is the `main` function. On x64 Linux systems, the first argument is conventionally passed in the register `rdi`. We can **p**rint it out with `p/x $rdi`: `0x555555555179` (`/x` prints out the value as hex). ASLR does not touch the bottom 12 bits of the address (also known as the page offset), and `179` matches what we found in Ghidra. We can now add a breakpoint directly on main by running `b *0x555555555179`. Alternatively, since `rdi` already points to `main`, we can run `b *$rdi` and let GDB expand the register to the correct address. To **c**ontinue program execution until the next breakpoint, use `c`.

Since we don't have symbol information for the local program, the next breakpoint message is not as helpful:
```c
Breakpoint 2, 0x0000555555555179 in ?? ()
```

At this point, our instruction pointer `rip` is right at the start of main. To e**x**amine the memory at this address, use `x $rip`. This will only print out the first word in hex, which is not very helpful. We can tell GDB to read and format the output as 30 **i**nstructions by specifying `x/30i $rip`:
```c
=> 0x555555555179:	push   rbp
   0x55555555517a:	mov    rbp,rsp
   0x55555555517d:	sub    rsp,0x40
   0x555555555181:	cmp    edi,0x2
   0x555555555184:	jne    0x5555555551f4
   0x555555555186:	mov    rax,QWORD PTR [rsi+0x8]
   0x55555555518a:	movabs rcx,0x7d336d31745f65
   0x555555555194:	push   rcx
   0x555555555195:	movabs rcx,0x313068775f336837
   0x55555555519f:	push   rcx
   0x5555555551a0:	movabs rcx,0x5f337233685f3534
   0x5555555551aa:	push   rcx
   0x5555555551ab:	movabs rcx,0x775f74317b67616c
   0x5555555551b5:	push   rcx
   0x5555555551b6:	mov    BYTE PTR [rsp-0x1],0x66
   0x5555555551bb:	sub    rsp,0x1
   0x5555555551bf:	mov    rdi,rax
   0x5555555551c2:	mov    rsi,rsp
   0x5555555551c5:	call   0x555555555040 <strcmp@plt>
   0x5555555551ca:	add    rsp,0x28
   0x5555555551ce:	test   eax,eax
   0x5555555551d0:	jne    0x5555555551f4
   0x5555555551d2:	movabs rax,0x2174636572726f
   0x5555555551dc:	push   rax
   0x5555555551dd:	mov    BYTE PTR [rsp-0x1],0x43
   0x5555555551e2:	sub    rsp,0x1
   0x5555555551e6:	mov    rdi,rsp
   0x5555555551e9:	call   0x555555555030 <puts@plt>
   0x5555555551ee:	add    rsp,0xc
   0x5555555551f2:	jmp    0x55555555521a
```
###### By default, GDB will use ATT syntax. You can fix this by running `set disassembly-flavor intel`, or putting it in `~/.gdbinit`.

Comparing these instructions with what we saw in Ghidra, we see that something has clearly changed `main`. Instead of the expected series of calls to `puts` and `sleep`, we see that the program builds a string on the stack, then compares it to our input with `strcmp`. We can see this more clearly if we decompile the `main` function at this point. We can load the new function into Ghidra by first dumping it to a binary file with `dump memory main.bin $rip $rip+170`, then loading `main.bin` as an x86-64 program.

```c

undefined8 FUN_00000000(int param_1,long param_2)

{
  int iVar1;
  undefined7 *puVar2;
  undefined1 local_69;
  undefined8 uStack_68;
  undefined8 uStack_60;
  undefined6 uStack_58;
  undefined2 uStack_52;
  undefined6 uStack_50;
  undefined1 local_4a;
  undefined1 uStack_49;
  undefined7 uStack_48;
  undefined1 auStack_41 [57];
  
  puVar2 = &uStack_48;
  if (param_1 == 2) {
    uStack_50 = 0x336d31745f65;
    local_4a = 0x7d;
    uStack_49 = 0;
    uStack_58 = 0x68775f336837;
    uStack_52 = 0x3130;
    uStack_60 = 0x5f337233685f3534;
    uStack_68 = 0x775f74317b67616c;
    local_69 = 0x66;
    iVar1 = func_0xfffffffffffffec7(*(undefined8 *)(param_2 + 8),&local_69);
    puVar2 = (undefined7 *)auStack_41;
    if (iVar1 == 0) {
      uStack_49 = 0x6f;
      uStack_48 = 0x217463657272;
      local_4a = 0x43;
      uStack_52 = 0x75;
      uStack_50 = 0;
      func_0xfffffffffffffeb7(&local_4a);
      return 0;
    }
  }
  *(undefined8 *)((long)puVar2 + -8) = 0x2174636572726f;
  *(undefined8 *)((long)puVar2 + -0x10) = 0x636e490000000000;
  *(undefined8 *)((long)puVar2 + -0x13) = 0x9d;
  func_0xfffffffffffffeb7((undefined1 *)((long)puVar2 + -0xb));
  return 0;
}
```

We can see that program compares argv[1] with a string, then prints a string if it matches and a different string if not. This structure matches what we should expect, but Ghidra's decompiler output is not very helpful here, since it interprets the stack strings as a series of large constants. While we could put effort into cleaning up the Ghidra output and statically recovering the strings, it will be easier to let the computer compute. When execution reaches `strcmp`, the flag must exist fully formed in the program state, so we should set a breakpoint on `strcmp` and continue up to that point.

```c
Breakpoint 3, 0x00007ffff7d83440 in ?? () from /usr/lib/libc.so.6
```

As before, we can e**x**amine the **s**trings pointed to by `strcmp`'s arguments with `x/s $rdi` and `x/s $rsi`. One of these will be the string we passed in, `flag{test}`, while the other will be the real flag.

###### Note that `p` prints the direct result of an expression, while `x` interprets the expression as an address and prints the contents of the memory it points to.






<hr style="border:1px solid white">

### 5 - Function Tracing
At this point, an experienced reverse engineer may remark that `strcmp` is a standard library function, and so we can recover its arguments by tracing dynamic function calls using a program like `ltrace`. This is in fact true. The following command's output will include the flag in plaintext for fast entry into CTFd.

In [ ]:
!ltrace ./chal1 flag{test}

Function tracing is a very powerful tool, and deserves its own notebook. To learn more, look into `LD_PRELOAD`, `ltrace`, `strace`, and `uftrace`.

Instruction-level debuggers like GDB, however, are more generally applicable and give us much more fine-grained flexibility in terms of the information we can learn from running a program.

<hr style="border:1px solid white">

### 6 - Test Your Skills
Now, try to solve `./chal2` using GDB. Function tracing will not help you here. You may find Ghidra helpful to get a rough idea of what the program does, but flag recovery will more easily be done in GDB.

<details> <summary> Hint: </summary> Where does the program compare your input to the flag?</details>

<hr style="border:1px solid white">

### 7 - Conclusion
In this notebook, we saw how we can use dynamic analysis in conjunction with static analysis to more efficiently uncover information about a program. Whereas static analysis tools only deal with the program as bytes on disk, dynamic analysis tools (such as debuggers) allow us to inspect transient program state not typically visible to a user. Both practices have their uses, and knowing when to use one or the other is a skill that comes with time. Learn how to use both, but avoid re-writing program code when running the original program is an option. If you ever ask yourself, "What is the value of this variable at this point in time?", then consider checking with a debugger.

<hr style="border:1px solid white">

### Bonus - Instruction Side-channel
A cursory look at `chal2` reveals that it executes a different number of instructions depending on the input.

In [ ]:
!perf stat -e instructions ./chal2 flag{test}

In [ ]:
!perf stat -e instructions ./chal2 flag{test}..............

In [ ]:
!perf stat -e instructions ./chal2 flaaaaaaaa..............

Combined with what we learned from using GDB and Ghidra, how might we use this information to recover the correct flag? Brute forcing a full flag is generally infeasible because of the number of different character combinations, but can knowing the instruction count corresponding to an input help optimize a search?

Write a script to automatically solve chal2 using this side-channel.

<hr style="border:1px solid white">